In [43]:
import requests
import time
import logging
import argparse
import json
import csv
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

In [44]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [45]:
def can_fetch(url, user_agent="*"):
    """Check robots.txt permission."""
    parsed = urljoin(url, "/robots.txt")
    rp = RobotFileParser()
    try:
        rp.set_url(parsed)
        rp.read()
        return rp.can_fetch(user_agent, url)
    except Exception as e:
        logging.warning(f"Could not read robots.txt for {url}: {e}")
        return True

In [46]:
def fetch_headlines(url, keyword=None, delay=2):
    """Fetch headlines from a news site."""
    if not can_fetch(url):
        logging.warning(f"Blocked by robots.txt: {url}")
        return []

    try:
        logging.info(f"Fetching {url}")
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        time.sleep(delay)  # polite delay

        soup = BeautifulSoup(resp.text, "html.parser")
        headlines = []

        # Example: look for <a> tags with headline text
        for tag in soup.find_all("a"):
            title = tag.get_text(strip=True)
            link = tag.get("href")
            if not title or not link:
                continue

            # Normalize URL
            full_url = urljoin(url, link)

            # Optional keyword filter
            if keyword and keyword.lower() not in title.lower():
                continue

            headlines.append({
                "title": title,
                "url": full_url,
                "time": None  # could extend to parse <time> tags
            })

        return headlines
    except Exception as e:
        logging.error(f"Error fetching {url}: {e}")
        return []

In [47]:
def save_results(results, output, fmt="json"):
  try:
    if fmt == "json":
      with open(output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    elif fmt == "csv":
      with open(output, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["title", "url", "time"])
        writer.writeheader()
        writer.writerows(results)
    logging.info(f"Resdults saved to {output}")
  except Exception as e:
    logging.error(f"Error saving results: {e}")

In [48]:
def main():
  logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
  parser = argparse.ArgumentParser(
    description="Web Scraper for Headlines")

  parser.add_argument("-u", "--urls", nargs=
"+", default=news_sources, help="List of site URLs")

  parser.add_argument("-o", "--output", default="headlines.json", help="Output file path")

  parser.add_argument("--fmt", choices=["json", "csv"], default= "json", help="Output format (json or csv)")

  parser.add_argument("--keyword", type=str, help="Filter headlines by keyword")

  parser.add_argument("--delay", type=int, help="Delay between requests (seconds)")

  # In a notebook environment, sys.argv often contains kernel-specific arguments.
  # Passing args=[] makes argparse use only default values and ignore sys.argv.
  args = parser.parse_args(args=[])

  all_results = []
  for url in args.urls:
    all_results.extend(fetch_headlines(url, keyword=args.keyword, delay=args.delay))

  save_results(all_results, args.output, fmt=args.fmt)

In [49]:
if __name__ == "__main__":
    main()

ERROR:root:Error fetching https://www.nytimes.com/: 'NoneType' object cannot be interpreted as an integer
ERROR:root:Error fetching https://www.bbc.com/news: 'NoneType' object cannot be interpreted as an integer
